## CS4287 Assignment 1: Robin vs. Not-Robin Classification with AlexNet and Mini-VGG
### 1. Setup and hyperparameters

In [1]:
# Group 11: Ciara Mollen (23376023), Ellis Mullen (21334498), Mohamed Amine Ait el Houssein (23397403)
# Code executes to the end without error: NO
# Third-party source: None


import os, json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

SEED = 42
IMG_SIZE = 128
BATCH_SIZE = 32
EPOCHS = 30
LEARNING_RATE = 0.001
N_FOLDS = 5

print("GPU:", tf.config.list_physical_devices('GPU'))

GPU: []


### 2. Loading the dataset split

In [ ]:
df = pd.read_csv("bird_dataset.csv")
df["filepath"] = df["filepath"].str.replace("\\", "/", regex=False)
df["fold"] = df["fold"].astype(str)
assert df["filepath"].map(os.path.exists).all(), "Some image paths were not found"

test_df = df[df["fold"] == "test"]
cv_df = df[df["fold"] != "test"]
print(len(cv_df), "images for cross-validation,", len(test_df), "held-out test images")

### 3. Input pipeline

In [ ]:
def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    return img, label

def make_ds(frame, shuffle):
    ds = tf.data.Dataset.from_tensor_slices((frame["filepath"].values, frame["label"].values))
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE).cache()
    if shuffle:
        ds = ds.shuffle(len(frame), seed=SEED)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

### 4. Model definitions

In [ ]:
def compile_model(model):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss='binary_crossentropy',
        metrics=['accuracy',
                 keras.metrics.Precision(name='precision'),
                 keras.metrics.Recall(name='recall')])
    return model

def build_vgg():
    model = keras.Sequential(name="mini_vgg")
    model.add(keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3)))
    model.add(layers.Rescaling(1./255))
    model.add(layers.Conv2D(32, kernel_size=(3, 3), use_bias=False, padding='same', kernel_initializer='he_normal'))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.Conv2D(32, kernel_size=(3, 3), use_bias=False, padding='same', kernel_initializer='he_normal'))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.MaxPool2D(pool_size=(2, 2)))
    model.add(layers.Conv2D(64, kernel_size=(3, 3), use_bias=False, padding='same', kernel_initializer='he_normal'))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.Conv2D(64, kernel_size=(3, 3), use_bias=False, padding='same', kernel_initializer='he_normal'))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.MaxPool2D(pool_size=(2, 2)))
    model.add(layers.Conv2D(128, kernel_size=(3, 3), use_bias=False, padding='same', kernel_initializer='he_normal'))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.Conv2D(128, kernel_size=(3, 3), use_bias=False, padding='same', kernel_initializer='he_normal'))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.MaxPool2D(pool_size=(2, 2)))
    model.add(layers.Conv2D(256, kernel_size=(3, 3), use_bias=False, padding='same', kernel_initializer='he_normal'))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.Conv2D(256, kernel_size=(3, 3), use_bias=False, padding='same', kernel_initializer='he_normal'))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.MaxPool2D(pool_size=(2, 2)))
    model.add(layers.GlobalAveragePooling2D())
    model.add(layers.Dense(256, activation='relu', kernel_initializer='he_normal'))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(1, activation='sigmoid', kernel_initializer='he_normal'))
    return compile_model(model)

def build_alexnet():
    model = keras.Sequential(name="alexnet")
    model.add(keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3)))
    model.add(layers.Rescaling(1./255))
    model.add(layers.Conv2D(filters=96, kernel_size=(11, 11), strides=4, use_bias=False, padding='same', kernel_initializer='he_normal'))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.MaxPooling2D(pool_size=(3, 3), strides=2))
    model.add(layers.Conv2D(filters=256, kernel_size=(5, 5), strides=1, use_bias=False, padding='same', kernel_initializer='he_normal'))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.MaxPooling2D(pool_size=(3, 3), strides=2))
    model.add(layers.Conv2D(filters=384, kernel_size=(3, 3), strides=1, use_bias=False, padding='same', kernel_initializer='he_normal'))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.Conv2D(filters=384, kernel_size=(3, 3), strides=1, use_bias=False, padding='same', kernel_initializer='he_normal'))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.Conv2D(filters=256, kernel_size=(3, 3), strides=1, use_bias=False, padding='same', kernel_initializer='he_normal'))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.MaxPooling2D(pool_size=(3, 3), strides=2))
    model.add(layers.Flatten())
    model.add(layers.Dense(1024, activation='relu'))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(1024, activation='relu'))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(1, activation='sigmoid'))
    return compile_model(model)

### 5. Stratified 5-fold cross-validation

In [ ]:
def run_cv(build_fn, name):
    histories, scores = [], []
    for k in range(N_FOLDS):
        keras.backend.clear_session()
        keras.utils.set_random_seed(SEED + k)

        train_ds = make_ds(cv_df[cv_df["fold"] != str(k)], shuffle=True)
        val_ds   = make_ds(cv_df[cv_df["fold"] == str(k)], shuffle=False)

        model = build_fn()
        h = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, verbose=2)

        histories.append(h.history)
        scores.append(model.evaluate(val_ds, return_dict=True, verbose=0))
        print(f"{name} fold {k}: {scores[-1]}")

    with open(f"{name}_cv_histories.json", "w") as f:
        json.dump(histories, f)
    return histories, pd.DataFrame(scores)

vgg_hist, vgg_scores = run_cv(build_vgg, "mini_vgg")
alex_hist, alex_scores = run_cv(build_alexnet, "alexnet")

### 6. Cross-validation results

In [ ]:
def mean_std(scores):
    return scores.mean().round(3).astype(str) + " ± " + scores.std().round(3).astype(str)

summary = pd.DataFrame({"Mini-VGG": mean_std(vgg_scores), "AlexNet": mean_std(alex_scores)})
print(summary)

def plot_cv(histories, name):
    metrics = ['loss', 'accuracy', 'precision', 'recall']
    plt.figure(figsize=(12, 8))
    for i, m in enumerate(metrics):
        plt.subplot(2, 2, i + 1)
        for key, style, label in [(m, '-', 'Training'), ('val_' + m, '--', 'Validation')]:
            arr = np.array([h[key] for h in histories])
            mean, std = arr.mean(axis=0), arr.std(axis=0)
            ep = np.arange(1, len(mean) + 1)
            plt.plot(ep, mean, style, label=label)
            plt.fill_between(ep, mean - std, mean + std, alpha=0.2)
        plt.title(m.capitalize())
        plt.xlabel('Epoch')
        plt.legend()
    plt.suptitle(f"{name}: mean ± std over {len(histories)} folds")
    plt.tight_layout()
    plt.show()

plot_cv(vgg_hist, "Mini-VGG")
plot_cv(alex_hist, "AlexNet")